In [ ]:
import os
%run common.py
import multiprocessing
from lxml import etree

# Loading

In [ ]:
graph_folder = '../../legal-networks-data/de/2_xml/'
files = list_dir(graph_folder, '.xml')

In [ ]:
len(files)

In [ ]:
def get_doc_attrs(file):
    tree = etree.parse(graph_folder + file)
    return dict(tree.xpath('/document[1]')[0].attrib)

with multiprocessing.Pool() as p:
    abks = p.map(get_doc_attrs, files)

In [ ]:
df = pd.DataFrame(abks)
df['abk'] = [k.split('_')[1] for k in df['key']]
df['abk_sort'] = [
    a.lower().replace('ß', 's').replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue') 
    for a in df.abk
]

# Create Complete List

In [ ]:
abk_heading_dict = {
    abk: 
    {
        df.iloc[idx]['heading']
        for idx in indices
        if not pd.isna(df.iloc[idx]['heading'])
    }
    for abk, indices in df.groupby('abk').indices.items()
}

In [ ]:
{
    abk: headings for abk, headings in abk_heading_dict.items()
    if len(headings) > 1
}

In [ ]:
tex = ''
md = 'Abkürzungsverzeichnis der Gesetze\n====================\n\n'

for abk in df.sort_values('abk_sort').abk.unique():
    if abk_heading_dict[abk]:
        heading = list(abk_heading_dict[abk])[0]
        tex += '\\item[{' + abk + '}] ' + heading + '\n\n'
        md += '- **' + abk + '** ' + heading + '\n'

In [ ]:
with open('../tables/de_gesetze_abks.tex', 'w') as f:
    f.write(tex)
    
with open('../tables/de_gesetze_abks.md', 'w') as f:
    f.write(md)